# W1 — Càrrega de dades i partició

Projecte 7: classificació d'imatges de menjar indi. Aquest notebook comprova l'estructura del conjunt de dades, carrega les rutes i etiquetes, inspecciona les imatges i crea una partició reproduïble de train/validació/test.

**Estructura esperada:** una carpeta per classe, amb les imatges a dins. Per exemple `data/IndianFood/samosa/imatge.jpg`. Ajusteu `ruta_dades` a la carpeta que conté les carpetes de classes.

In [ ]:
from pathlib import Path
from collections import Counter

from PIL import Image
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
import pandas as pd

SEED = 42
EXTENSIONS = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}
ruta_dades = Path('dataset/Food Classification')  # Carpeta amb les 20 classes

if not ruta_dades.exists():
    raise FileNotFoundError(f'No existeix la carpeta: {ruta_dades.resolve()}')

print(f'Carpeta de dades: {ruta_dades.resolve()}')

## 1. Llegir les rutes i les etiquetes

Cada subcarpeta immediata es considera una classe. En aquesta primera passada només llegim metadades i rutes; encara no carreguem totes les imatges a memòria.

In [ ]:
rutes_imatges = []
etiquetes = []

carpetes_classes = sorted(p for p in ruta_dades.iterdir() if p.is_dir())
if not carpetes_classes:
    raise ValueError('No s’han trobat subcarpetes de classes. Reviseu ruta_dades i l’estructura del conjunt.')

for carpeta_classe in carpetes_classes:
    for ruta in sorted(carpeta_classe.rglob('*')):
        if ruta.is_file() and ruta.suffix.lower() in EXTENSIONS:
            rutes_imatges.append(ruta)
            etiquetes.append(carpeta_classe.name)

if not rutes_imatges:
    raise ValueError('No s’han trobat fitxers d’imatge. Comproveu la ruta i les extensions.')

print(f'Nombre d’imatges: {len(rutes_imatges)}')
print(f'Nombre de classes: {len(set(etiquetes))}')
print('Classes:', sorted(set(etiquetes)))

## 2. Distribució de classes i comprovació d’imatges

Comprovem si els fitxers es poden obrir i mirem quantes imatges hi ha per classe. Els fitxers corruptes s’exclouran de la partició i quedaran identificats a `imatges_no_valides`.

In [ ]:
taula_classes = pd.Series(Counter(etiquetes), name="nombre_imatges").sort_values(ascending=False)
display(taula_classes.to_frame())

## 3. Mostra visual

Mostrem una imatge de cada classe per confirmar que les etiquetes corresponen al contingut.

In [ ]:
classes = sorted(set(etiquetes))
fig, eixos = plt.subplots(len(classes), 1, figsize=(5, max(3, 3 * len(classes))))
if len(classes) == 1:
    eixos = [eixos]
for eix, classe in zip(eixos, classes):
    index = etiquetes.index(classe)
    with Image.open(rutes_imatges[index]) as imatge:
        eix.imshow(imatge.convert('RGB'))
    eix.set_title(classe)
    eix.axis('off')
plt.tight_layout()
plt.show()

## 4. Crear train/validació/test

Fem una partició estratificada aproximada de 70%/15%/15%. La llavor fixa fa que la partició es pugui reproduir. **Abans d’interpretar els resultats**, cal revisar possibles fotos duplicades o gairebé duplicades: si n’hi ha, haurien de quedar al mateix subconjunt per evitar una estimació massa optimista.

In [ ]:
rutes_train, rutes_temp, etiquetes_train, etiquetes_temp = train_test_split(
    rutes_imatges, etiquetes, test_size=0.30, random_state=SEED, stratify=etiquetes
)
rutes_val, rutes_test, etiquetes_val, etiquetes_test = train_test_split(
    rutes_temp, etiquetes_temp, test_size=0.50, random_state=SEED, stratify=etiquetes_temp
)

print(f'Train: {len(rutes_train)} ({len(rutes_train)/len(rutes_imatges):.1%})')
print(f'Validació: {len(rutes_val)} ({len(rutes_val)/len(rutes_imatges):.1%})')
print(f'Test: {len(rutes_test)} ({len(rutes_test)/len(rutes_imatges):.1%})')

files = []
for particio, rutes, noms_classes in [
    ('train', rutes_train, etiquetes_train),
    ('validacio', rutes_val, etiquetes_val),
    ('test', rutes_test, etiquetes_test),
]:
    files.extend({'ruta': str(ruta), 'classe': classe, 'particio': particio}
                 for ruta, classe in zip(rutes, noms_classes))
manifest = pd.DataFrame(files)
display(pd.crosstab(manifest['classe'], manifest['particio']))

## 5. Desar la partició

Desem un manifest amb la ruta, la classe i la partició perquè la resta del projecte utilitzi exactament el mateix split. El directori `data/splits` es crea si encara no existeix.

In [ ]:
directori_splits = Path('data/splits')
directori_splits.mkdir(parents=True, exist_ok=True)
ruta_manifest = directori_splits / 'split_w1.csv'
manifest.to_csv(ruta_manifest, index=False)
print(f'Manifest desat a: {ruta_manifest.resolve()}')

## Resultat del checkpoint W1

En acabar, anoteu al README: (1) quantes imatges vàlides i classes s’han carregat, (2) la mida de cada partició i la llavor, (3) problemes observats, inclosos duplicats, classes desequilibrades o imatges corruptes. Aquest notebook encara no entrena el classificador; el pas següent és implementar una baseline simple sobre aquest split.